Download [dataset](https://www.kaggle.com/competitions/classification-of-butterflies)

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

torch.random.manual_seed(42)
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"
learning_rate = 1e-3
batch_size = 64
epochs = 1000

device

device(type='mps')

In [21]:
import os
import cv2
from torch.utils.data import Dataset, DataLoader, random_split

def normalize(images): # SHWC -> SCHW
    images = np.transpose(images, (0, 3, 1, 2))
    mean = np.mean(images, axis=(1, 2, 3), keepdims=True)
    std = np.std(images, axis=(1, 2, 3), keepdims=True)
    return (images - mean) / std

class TrainButterflyDataset(Dataset):
    def __init__(self):
        self.data = []
        self.labels = []
        for folder in os.listdir("data/train_split/"):
            class_name = int(folder[folder.rfind("_")+1:])
            for file in os.listdir(f"data/train_split/{folder}"):
                image = cv2.imread(f"data/train_split/{folder}/{file}")
                np_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                self.data.append(np_image)
                self.labels.append(class_name)
        
        self.data = normalize(np.array(self.data, dtype=np.float32))
        self.labels = np.array(self.labels, dtype=np.int64)

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

class TestButterflyDataset(Dataset):
    def __init__(self):
        self.data = []
        for file in os.listdir(f"data/valid/"):
            image = cv2.imread(f"data/valid/{file}")
            np_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB).astype(np.float32)
            np_image = np.transpose(np_image, (2, 0, 1))  # HWC -> CHW
            self.data.append(np_image)
        self.data = np.array(self.data, dtype=np.float32)

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

full_train_dataset = TrainButterflyDataset()
train_dataset, valid_dataset = random_split(full_train_dataset, [0.8, 0.2])
test_dataset = TestButterflyDataset()

full_train_loader = DataLoader(full_train_dataset, batch_size=64, shuffle=True)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [22]:
full_train_dataset[0][0].shape

(3, 224, 224)

In [23]:
import torch.nn as nn
import torch.optim as optim

def conv_block(in_c, out_c):
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU(),
        nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )

class ButterflyNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, 32),
            conv_block(32, 64),
            conv_block(64, 128),
            conv_block(128, 256),
            conv_block(256, 512),
        )
        self.head = nn.Sequential(
            self.features,
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.head(x)


net = ButterflyNet(49).to(device)
optimizer = optim.Adam(net.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
scheduler = lambda opt: optim.lr_scheduler.ReduceLROnPlateau(opt, "min", patience=10, factor=0.1)

In [24]:
from trainer import Trainer

trainer = Trainer(net, criterion, optimizer, device, epoch_amount=epochs, scheduler=scheduler)
trainer.fit(train_loader, valid_loader)

Эпоха: 0 Loss_train: 3.168518239451993, 0:00:49.263317 сек
Loss_val: 3.01752907037735

Эпоха: 1 Loss_train: 2.417496835031817, 0:00:47.151159 сек
Loss_val: 2.586160257458687

Эпоха: 2 

KeyboardInterrupt: 

In [ ]:
torch.save(trainer.best_model.state_dict(), "data/butterfly_net.pt")

In [ ]:
pred = trainer.predict(test_loader)
classes = pred.softmax(dim=1).argmax(dim=1).numpy()
pd.DataFrame({"index": np.arange(0, len(classes)), "label": classes}).to_csv(
    "data/predictions.csv", index=False
)
